In [ ]:
!pip install datasets

In [ ]:
!pip install transformers

In [ ]:
from datasets import load_dataset

dataset = load_dataset("sevenreasons/genius-lyrics-russian")

README.md:   0%|          | 0.00/106 [00:00<?, ?B/s]

genius-ru.json: reconstructing file:   0%|          |  0.00B /  541MB            

genius-ru.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/192020 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd

data = pd.DataFrame(dataset['train'])

In [ ]:
import re

def cleaner(text):
  if not isinstance(text, str):
    return ''

  text = text.lower()

  text = re.sub(r'\[.*?\]', '', text)
  text = re.sub(r'\(.*?\)', '', text)

  text = re.sub(r'[^a-zа-яё0-9\s.,!?\-]', ' ', text)
  text = re.sub(r'[ \t]+', ' ', text)
  text = re.sub(r'\n{3,}', '\n\n', text)

  return text.strip()

data['lyrics'] = data['lyrics'].apply(cleaner)

data = data[data['lyrics'] != ""]

In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

class Tokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.vocab_size = len(tokenizer)

    def encode(self, text):
        return self.tokenizer.encode(text, add_special_tokens=False)

    def decode(self, nums):
        return self.tokenizer.decode(nums)

tokenizer = Tokenizer(tokenizer)
print(f"vocab size = {tokenizer.vocab_size}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

vocab size = 50257


In [ ]:
import random
import torch

def get_batch():
    xs, ys = [], []

    while len(xs) < batch_size:
        random_song = random.choice(data['lyrics'].values)

        encoded_song = tokenizer.encode(random_song)

        if len(encoded_song) <= block_size:
            continue

        start_idx = random.randint(0, len(encoded_song) - block_size - 1)

        x_chunk = encoded_song[start_idx : start_idx + block_size]
        y_chunk = encoded_song[start_idx + 1 : start_idx + block_size + 1]

        xs.append(torch.tensor(x_chunk, dtype=torch.long))
        ys.append(torch.tensor(y_chunk, dtype=torch.long))

    x = torch.stack(xs)
    y = torch.stack(ys)

    return x, y

xb, yb = get_batch()
print(f"x.shape = {xb.shape}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1101 > 1024). Running this sequence through the model will result in indexing errors


x.shape = torch.Size([32, 128])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
  def __init__(self, embedding_dim, head_size, block_size):
    super().__init__()
    self.query = nn.Linear(embedding_dim, head_size, bias=False)
    self.key   = nn.Linear(embedding_dim, head_size, bias=False)
    self.value = nn.Linear(embedding_dim, head_size, bias=False)

    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B, T, C = x.shape
    q = self.query(x)
    k = self.key(x)
    v = self.value(x)

    wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)

    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

    wei = F.softmax(wei, dim=-1)

    out = wei @ v

    return out

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, embedding_dim, block_size):
    super().__init__()
    head_size = embedding_dim // num_heads

    self.heads = nn.ModuleList([Head(embedding_dim, head_size, block_size) for _ in range(num_heads)])
    self.proj = nn.Linear(embedding_dim, embedding_dim)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)

    return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
  def __init__(self, dim, eps=1e-6):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.ones(dim))

  def forward(self, x):
    variance = x.pow(2).mean(-1, keepdim=True)
    return x * torch.rsqrt(variance + self.eps) * self.weight

class FeedForward(nn.Module):
  def __init__(self, embedding_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(embedding_dim, 4 * embedding_dim),
        nn.GELU(),
        nn.Linear(4 * embedding_dim, embedding_dim),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, embedding_dim, num_heads, block_size):
    super().__init__()
    self.sa = MultiHeadAttention(num_heads, embedding_dim, block_size)
    self.ffwd = FeedForward(embedding_dim)
    self.ln1 = RMSNorm(embedding_dim)
    self.ln2 = RMSNorm(embedding_dim)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))

    return x


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RapLLM(nn.Module):
  def __init__(self, vocab_size, embedding_dim, block_size, num_layers, num_heads):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)
    self.position_embedding_table = nn.Embedding(block_size, embedding_dim)

    self.blocks = nn.Sequential(*[Block(embedding_dim, num_heads, block_size) for _ in range(num_layers)])

    self.ln_f = RMSNorm(embedding_dim)

    self.lm_head = nn.Linear(embedding_dim, vocab_size)

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, idx, targets=None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)
    pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
    x = tok_emb + pos_emb

    x = self.blocks(x)

    x = self.ln_f(x)
    logits = self.lm_head(x)

    loss = None
    if targets is not None:
        B, T, C = logits.shape
        logits_flat = logits.view(B * T, C)
        targets_flat = targets.view(B * T)
        loss = F.cross_entropy(logits_flat, targets_flat)

    return logits, loss

  def generate(self, idx, max_new_tokens, temperature=0.8):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]

      logits, _ = self(idx_cond)

      logits = logits[:, -1, :]

      logits = logits / temperature

      probs = F.softmax(logits, dim=-1)

      idx_next = torch.multinomial(probs, num_samples=1)

      idx = torch.cat((idx, idx_next), dim=1)

    return idx


In [ ]:
import torch
import torch.optim as optim

vocab_size = tokenizer.vocab_size
device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 128
batch_size = 32
embedding_dim = 512
num_layers = 24
num_heads = 8

model = RapLLM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    block_size=block_size,
    num_layers=num_layers,
    num_heads=num_heads
).to(device)


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)

max_iters = 50_000
eval_interval = 200

model.train()

for steps in range(max_iters):

  xb, yb = get_batch()
  xb, yb = xb.to(device), yb.to(device)

  logits, loss = model(xb, yb)

  optimizer.zero_grad(set_to_none=True)

  loss.backward()

  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

  optimizer.step()

  if steps % eval_interval == 0:
      print(f"Шаг {steps} | Текущий Loss = {loss.item():.4f}")

print(f" Обучение завершено! Итоговый Loss = {loss.item():.4f}")


Шаг 0 | Текущий Loss = 1.4817
Шаг 200 | Текущий Loss = 1.4132
Шаг 400 | Текущий Loss = 1.4835
Шаг 600 | Текущий Loss = 1.5253
Шаг 800 | Текущий Loss = 1.4467
Шаг 1000 | Текущий Loss = 1.4106
Шаг 1200 | Текущий Loss = 1.5142
Шаг 1400 | Текущий Loss = 1.3845
Шаг 1600 | Текущий Loss = 1.5075
Шаг 1800 | Текущий Loss = 1.4277
Шаг 2000 | Текущий Loss = 1.4024
Шаг 2200 | Текущий Loss = 1.4537
Шаг 2400 | Текущий Loss = 1.3780
Шаг 2600 | Текущий Loss = 1.3256
Шаг 2800 | Текущий Loss = 1.5737
Шаг 3000 | Текущий Loss = 1.3866
Шаг 3200 | Текущий Loss = 1.4740
Шаг 3400 | Текущий Loss = 1.3696
Шаг 3600 | Текущий Loss = 1.3734
Шаг 3800 | Текущий Loss = 1.2894


KeyboardInterrupt: 

In [ ]:
model = RapLLM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    block_size=block_size,
    num_layers=num_layers,
    num_heads=num_heads
)

state_dict = torch.load('/content/rap_model.pt', map_location='cpu', weights_only=False)

model.load_state_dict(state_dict)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

RapLLM(
  (token_embedding_table): Embedding(50257, 512)
  (position_embedding_table): Embedding(128, 512)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x Head(
            (query): Linear(in_features=512, out_features=64, bias=False)
            (key): Linear(in_features=512, out_features=64, bias=False)
            (value): Linear(in_features=512, out_features=64, bias=False)
          )
        )
        (proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
      (ln1): RMSNorm()
      (ln2): RMSNorm()
    )
    (1): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x Head(
            (query): Linear(in_features=

In [ ]:
model.eval()

prompt = "влетел на бит"

tokens = tokenizer.encode(prompt)
context = torch.tensor([tokens], dtype=torch.long, device=device)

with torch.no_grad():
  tokens = model.generate(context, max_new_tokens=300, temperature=0.8)

  res = tokens[0].tolist()

  rap = tokenizer.decode(res)

  print(rap)


влетел на бите
мне забиты таким минут
им найти как шагу

всешь, я знаю всё потерял спокойной каверти 
попробуй выйти из неё 
и кто все выпадет в мои мысли даже не смогу

я всё, что ты скажешь мне действительно 
я всё, что такое рассеговоруются
я на тебя не застанусь, но всё тепла не пр


In [ ]:
torch.save(model.state_dict(), 'rap_model3.pt')